# EDA — NIST Cybersecurity Training

**Dataset:** [`ethanolivertroy/nist-cybersecurity-training`](https://huggingface.co/datasets/ethanolivertroy/nist-cybersecurity-training)

≈530 000 ejemplos en formato chat extraídos de 596 publicaciones NIST (FIPS, SP 800/1800, IR, CSWP).

**Formato:** JSON Lines (`train.jsonl`, `valid.jsonl`): `messages` (lista system/user/assistant) + `metadata` (`source`, `type`, `chunk_id`). La carpeta `embeddings/` (~10 GB) **no se usa** en este EDA.

**Preguntas guía para el EDA**
- ¿Qué publicaciones NIST aportan más ejemplos?
- ¿Cómo se reparten los tipos (sección, chunk, control, definición)?
- ¿Qué tan largos son los contenidos por tipo?
- ¿Hay preguntas duplicadas o fuga entre train y valid?

In [ ]:
import sys, json, re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

# Busca la raíz del repo (donde está src/profiler.py) y la agrega al path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "profiler.py").exists())
sys.path.insert(0, str(ROOT))
from src.profiler import *

sns.set_theme(style="whitegrid", palette="tab10")
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 120)

ctx = eda_context("10")   # rutas: datos en data/raw/..., resultados en esta carpeta
print(ctx)

## 1. Inventario (sin la carpeta de embeddings)

In [ ]:
files = list_files(ctx.data_dir)
display(files[~files["archivo"].str.startswith("embeddings/")])
print("Embeddings (no se cargan):", round(files[files["archivo"].str.startswith("embeddings/")]["size_mb"].sum() / 1e3, 2), "GB")
for f in ["data_stats.json", "embeddings/embedding_summary.json"]:
    p = ctx.data_dir / f
    if p.exists():
        print(f"\n{f}:"); print(json.dumps(read_json_any(p), indent=2))

## 2. Cargar muestras de train y valid
Los archivos pesan ~800 MB y ~200 MB: se toma una muestra aleatoria.

In [ ]:
SAMPLE_N = 30_000

def parse_nist(d):
    """Convierte messages/metadata en columnas planas."""
    out = pd.DataFrame({
        role: d["messages"].apply(lambda ms: next((m.get("content") for m in ms if m.get("role") == role), None))
        for role in ["system", "user", "assistant"]
    })
    meta = pd.json_normalize(d["metadata"].apply(lambda x: x if isinstance(x, dict) else {}))
    return pd.concat([out, meta], axis=1)

print("Primer registro:"); display(peek_record(ctx.data_dir / "train.jsonl"))
N_TRAIN = count_lines(ctx.data_dir / "train.jsonl")
N_VALID = count_lines(ctx.data_dir / "valid.jsonl")
print(f"train: {N_TRAIN:,} | valid: {N_VALID:,}")

df = parse_nist(read_jsonl(ctx.data_dir / "train.jsonl", sample=SAMPLE_N))
dv = parse_nist(read_jsonl(ctx.data_dir / "valid.jsonl", sample=SAMPLE_N // 4))
print(df.shape, dv.shape)
display(df.head(3))
display(profile_df(df))

## 3. Tipos y publicaciones fuente

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
df["type"].value_counts().plot.bar(ax=axes[0]); axes[0].set_title("Tipo de ejemplo (muestra train)")
df["source"].value_counts().head(20).plot.barh(ax=axes[1]); axes[1].invert_yaxis(); axes[1].set_title("Top 20 publicaciones")
plt.tight_layout(); save_fig(fig, ctx, "tipos_y_fuentes"); plt.show()
print("Publicaciones distintas en la muestra:", df["source"].nunique())

serie = df["source"].str.extract(r"^NIST\s+(\w+)")[0]
display(serie.value_counts().to_frame("n"))

## 4. Largo del contenido

In [ ]:
df["len_assistant"] = df["assistant"].fillna("").str.len()
fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(data=df, x="type", y="len_assistant", showfliers=False, ax=ax); ax.set_title("Largo de la respuesta por tipo")
save_fig(fig, ctx, "largo_por_tipo"); plt.show()
display(df.groupby("type")["len_assistant"].describe().round(0))
df = df.drop(columns=["len_assistant"])

## 5. Duplicados y fuga train/valid

In [ ]:
display(duplicates_report(df, {"user": ["user"], "assistant": ["assistant"], "user + assistant": ["user", "assistant"]}))
print("System prompts distintos:", df["system"].nunique())
display(df["user"].str.split().str[:4].str.join(" ").value_counts().head(15).to_frame("n"))

ht, hv = set(df["assistant"].map(text_hash)), set(dv["assistant"].map(text_hash))
print(f"Respuestas de la muestra de valid presentes en la muestra de train: {len(ht & hv)} de {len(hv)}")
print("(Es una cota inferior: con muestras solo se detecta parte del solapamiento real)")

## 6. Ejemplos

In [ ]:
show_examples(df, n=3, cols=["source", "type", "user", "assistant"], max_chars=600)

## Guardar resultados
Se guardan en `tables/` de esta carpeta. `resumen.csv` lo consolida `eda/00_general`.

In [ ]:
save_table(profile_df(df), ctx, "perfil_columnas")
display(save_summary(ctx, "ethanolivertroy/nist-cybersecurity-training", df, total_rows=N_TRAIN + N_VALID, dup_cols=["user", "assistant"],
                     notas="muestra aleatoria de train"))

## Próximos pasos / notas
- Anota tus hallazgos en el `README.md` de esta carpeta.
-